# Pandas 02 — Cleaning: timestamps, duplicates, text numbers, sentinels, gaps

Each step is shown on a tiny table first, then applied to `../data/hourly_power_raw.csv`.
At the end the cleaned file is compared with the known-good `hourly_power_clean.csv`.

**What's in here**
1. timestamps first
2. duplicates: exact rows vs duplicate keys
3. numbers stored as text
4. sentinel values
5. constant columns
6. reindex to a complete grid
7. missing-value strategies
8. fill with a group statistic
9. `dropna`: `subset=` vs `how=`
10. outliers: clip vs winsorise vs flag
11. final checks as `assert`s
12. compare against the known-good file
13. string clean-up, 14. booleans / dates / categoricals read as text

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

In [2]:
raw = pd.read_csv("../data/hourly_power_raw.csv")
raw.head()

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,region
0,2023-03-30 23:00:00,28555.6,4.97,8.75,0.0,70.16,GB
1,2023-07-16 15:00:00,27602.0,21.96,6.52,523.2,23.19,GB
2,2022-12-25 16:00:00,33773.8,4.15,9.57,18.5,115.79,GB
3,2023-03-10 01:00:00,26151.0,2.06,5.83,0.0,80.50,GB
4,2023-11-09 05:00:00,24584.6,6.58,4.76,0.0,69.49,GB


## 1. Timestamps first

Convert to real timestamps, then sort. Everything later (`shift`, `rolling`, gaps)
assumes sorted, unique timestamps.

In [3]:
t = pd.DataFrame({"time": ["2023-01-01 02:00", "2023-01-01 00:00", "2023-01-01 01:00"],
                  "v": [3, 1, 2]})
t

,time,v
0,2023-01-01 02:00,3
1,2023-01-01 00:00,1
2,2023-01-01 01:00,2


In [4]:
t["time"] = pd.to_datetime(t["time"], utc=True)
t = t.sort_values("time")
t

,time,v
1,2023-01-01 00:00:00+00:00,1
2,2023-01-01 01:00:00+00:00,2
0,2023-01-01 02:00:00+00:00,3


`utc=True` makes the timestamps timezone-aware UTC. Row order is now by time (the
original row numbers 1, 2, 0 stay as the index until we reset it).

In [5]:
df = raw.copy()
df["time"] = pd.to_datetime(df["time"], utc=True)
df = df.sort_values("time").reset_index(drop=True)
df.head(3)

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,region
0,2022-01-01 00:00:00+00:00,26858.4,0.11,7.00,0.0,81.83,GB
1,2022-01-01 01:00:00+00:00,26177.8,-0.18,6.61,0.0,88.21,GB
2,2022-01-01 02:00:00+00:00,26229.4,-1.11,7.14,0.0,84.71,GB


## 2. Duplicates: exact rows vs duplicate keys

An exact duplicate repeats every column. A duplicate *key* repeats the timestamp but
may disagree in the values — that one needs a decision.

In [6]:
d = pd.DataFrame({"time": ["t1", "t2", "t2", "t3"], "v": [1, 2, 2, 3]})
d

,time,v
0,t1,1
1,t2,2
2,t2,2
3,t3,3


In [7]:
d.drop_duplicates()

,time,v
0,t1,1
1,t2,2
3,t3,3


Row 2 was an exact copy of row 1, so it is gone. Now a case where the key repeats but
the value differs:

In [8]:
d2 = pd.DataFrame({"time": ["t1", "t2", "t2", "t3"], "v": [1, 2, 5, 3]})
d2

,time,v
0,t1,1
1,t2,2
2,t2,5
3,t3,3


In [9]:
d2.drop_duplicates()          # nothing removed: the rows are not identical

,time,v
0,t1,1
1,t2,2
2,t2,5
3,t3,3


In [10]:
d2.drop_duplicates(subset=["time"], keep="last")   # decide: keep the last one per key

,time,v
0,t1,1
2,t2,5
3,t3,3


On the real file: count both kinds, then drop by key.

In [11]:
print("exact duplicate rows :", df.duplicated().sum())
print("duplicate timestamps :", df.duplicated(subset=["time"]).sum())

exact duplicate rows : 15
duplicate timestamps : 15


In [12]:
dup_times = df.loc[df.duplicated(subset=["time"], keep=False)]
dup_times.head(4)

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,region
820,2022-02-04 07:00:00+00:00,32724.5,0.53,5.42,41.5,127.95,GB
821,2022-02-04 07:00:00+00:00,32724.5,0.53,5.42,41.5,127.95,GB
1566,2022-03-07 08:00:00+00:00,36737.5,-1.14,6.74,157.4,110.76,GB
1567,2022-03-07 08:00:00+00:00,36737.5,-1.14,6.74,157.4,110.76,GB


These pairs agree in every column, so keeping either is fine.

In [13]:
df = df.drop_duplicates(subset=["time"], keep="last").reset_index(drop=True)
print(len(df), "rows; unique times:", df["time"].is_unique)

17442 rows; unique times: True


## 3. Numeric columns stored as text

`astype(float)` stops at the first bad value. `pd.to_numeric(errors="coerce")` turns
bad values into NaN, so you can count and inspect them.

In [14]:
p = pd.Series(["70.1", "23.2", "missing", "115.8"])
p

0       70.1
1       23.2
2    missing
3      115.8
dtype: object

In [15]:
try:
    p.astype(float)
except ValueError as e:
    print("ValueError:", e)

ValueError: could not convert string to float: 'missing'


In [16]:
pd.to_numeric(p, errors="coerce")

0     70.1
1     23.2
2      NaN
3    115.8
dtype: float64

On the real file: convert, then count how many became NaN that were not NaN before.

In [17]:
before = df["price_eur_mwh"].isna().sum()
df["price_eur_mwh"] = pd.to_numeric(df["price_eur_mwh"], errors="coerce")
after = df["price_eur_mwh"].isna().sum()
print("NaN before:", before, " after:", after, " dtype:", df["price_eur_mwh"].dtype)

NaN before: 0  after: 100  dtype: float64


## 4. Sentinel values

`-999` is a placeholder someone typed instead of NaN. `describe()` reveals it in `min`;
`mask` turns it into NaN.

In [18]:
s = pd.Series([5.0, -999.0, 7.0, -999.0])
s

0      5.0
1   -999.0
2      7.0
3   -999.0
dtype: float64

In [19]:
s.mask(s <= -100)

0    5.0
1    NaN
2    7.0
3    NaN
dtype: float64

In [20]:
print("sentinels in temp_c:", (df["temp_c"] <= -100).sum())
df["temp_c"] = df["temp_c"].mask(df["temp_c"] <= -100)
df["temp_c"].describe().round(1)

sentinels in temp_c: 63


count    17230.0
mean         9.9
std          6.7
min         -6.4
25%          4.5
50%         10.0
75%         15.3
max         27.7
Name: temp_c, dtype: float64

`min` is now a plausible temperature.

## 5. Drop constant / useless columns

A column with one distinct value carries no information.

In [21]:
c = pd.DataFrame({"region": ["GB", "GB", "GB"], "v": [1, 2, 3]})
c.nunique()

region    1
v         3
dtype: int64

In [22]:
c.drop(columns=["region"])

,v
0,1
1,2
2,3


In [23]:
print(df.nunique())
df = df.drop(columns=["region"])

time               17442
consumption_mwh    16497
temp_c              2843
wind_ms             1366
solar_wm2           4062
price_eur_mwh       9931
region                 1
dtype: int64


## 6. Reindex to a complete hourly grid

Set the timestamp as index, build the full grid, `reindex`. Missing hours appear as NaN rows.

In [24]:
h = pd.DataFrame({"v": [1.0, 2.0, 4.0]},
                 index=pd.to_datetime(["2023-01-01 00:00", "2023-01-01 01:00", "2023-01-01 03:00"]))
h

,v
2023-01-01 00:00:00,1.0
2023-01-01 01:00:00,2.0
2023-01-01 03:00:00,4.0


In [25]:
grid = pd.date_range("2023-01-01 00:00", "2023-01-01 03:00", freq="h")
grid

DatetimeIndex(['2023-01-01 00:00:00', '2023-01-01 01:00:00', '2023-01-01 02:00:00', '2023-01-01 03:00:00'], dtype='datetime64[ns]', freq='h')

In [26]:
h.reindex(grid)

,v
2023-01-01 00:00:00,1.0
2023-01-01 01:00:00,2.0
2023-01-01 02:00:00,NaN
2023-01-01 03:00:00,4.0


02:00 was missing in the file and now shows as a NaN row. On the real data:

In [27]:
df = df.set_index("time")
grid = pd.date_range(df.index.min(), df.index.max(), freq="h")
print("rows in file:", len(df), " grid hours:", len(grid))
df = df.reindex(grid)
df.index.name = "time"
print("missing hours:", df["consumption_mwh"].isna().sum())

rows in file: 17442  grid hours: 17520
missing hours: 78


Which days are missing hours? Group the NaN rows by date.

In [28]:
missing = df[df["consumption_mwh"].isna()]
missing.groupby(missing.index.date).size().sort_values(ascending=False).head(5)

2022-03-27    24
2022-06-07     2
2022-01-07     1
2023-08-11     1
2023-01-27     1
dtype: int64

2022-03-27 is missing all 24 hours; the other days one or two hours each.

## 7. Missing-value strategies

The same 5-value column under each strategy, so you can compare.

In [29]:
x = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan], index=["t1", "t2", "t3", "t4", "t5"])
x

t1    1.0
t2    NaN
t3    NaN
t4    4.0
t5    NaN
dtype: float64

In [30]:
pd.DataFrame({
    "x": x,
    "ffill": x.ffill(),
    "ffill(limit=1)": x.ffill(limit=1),
    "bfill": x.bfill(),
    "interpolate": x.interpolate(),
    "fillna(0)": x.fillna(0),
})

,x,ffill,ffill(limit=1),bfill,interpolate,fillna(0)
t1,1.0,1.0,1.0,1.0,1.0,1.0
t2,NaN,1.0,1.0,4.0,2.0,0.0
t3,NaN,1.0,NaN,4.0,3.0,0.0
t4,4.0,4.0,4.0,4.0,4.0,4.0
t5,NaN,4.0,4.0,NaN,4.0,0.0


- `ffill` copies the last seen value forward (t5 gets 4.0; stale but plausible for a slow feature)
- `ffill(limit=1)` only fills one step, so t3 stays NaN
- `bfill` copies from the *future* (t2 gets 4.0) — never for a forecasting feature
- `interpolate` draws a line between 1 and 4
- `fillna(0)` invents a value

**Pitfall:** never fill the column you are trying to predict. A filled target teaches
the model a value that never happened. Fill features carefully, drop target rows.

In [31]:
df["temp_c"] = df["temp_c"].interpolate(limit=3)
df["price_eur_mwh"] = df["price_eur_mwh"].ffill(limit=2)
df.isna().sum()

consumption_mwh    78
temp_c             21
wind_ms            78
solar_wm2          78
price_eur_mwh      22
dtype: int64

## 8. Fill with a group statistic

Fill each NaN with the median of its group (here: hour of day).

In [32]:
g = pd.DataFrame({"hour": [0, 0, 1, 1, 0], "v": [10.0, np.nan, 20.0, 22.0, 12.0]})
g

,hour,v
0,0,10.0
1,0,NaN
2,1,20.0
3,1,22.0
4,0,12.0


In [33]:
g.groupby("hour")["v"].transform("median")

0    11.0
1    11.0
2    21.0
3    21.0
4    11.0
Name: v, dtype: float64

`transform` gives one value per row: the median of that row's group (hour 0 → 11, hour 1 → 21).
`fillna` then only touches the NaN.

In [34]:
g["v"] = g["v"].fillna(g.groupby("hour")["v"].transform("median"))
g

,hour,v
0,0,10.0
1,0,11.0
2,1,20.0
3,1,22.0
4,0,12.0


In [35]:
hour_median = df.groupby(df.index.hour)["temp_c"].transform("median")
df["temp_c"] = df["temp_c"].fillna(hour_median)
print("temp NaN left:", df["temp_c"].isna().sum())

temp NaN left: 0


## 9. `dropna`: `subset=` vs `how=`

`how="all"` drops rows where every value is NaN; `subset=` drops rows where the named columns are NaN.

In [36]:
r = pd.DataFrame({"a": [1.0, np.nan, np.nan], "b": [1.0, 2.0, np.nan]})
r

,a,b
0,1.0,1.0
1,NaN,2.0
2,NaN,NaN


In [37]:
r.dropna(how="all")

,a,b
0,1.0,1.0
1,NaN,2.0


In [38]:
r.dropna(subset=["a"])

,a,b
0,1.0,1.0


On the real data, the missing hours are NaN in every column; keep them for now
(the model step will drop rows where the target is NaN).

## 10. Outliers: clip vs winsorise vs flag

Three treatments of one spike.

In [39]:
o = pd.Series([10, 11, 200, 12, 9])
lo, hi = o.quantile(0.1), o.quantile(0.9)
print("10% / 90% quantiles:", lo, hi)
pd.DataFrame({
    "o": o,
    "clip(upper=50)": o.clip(upper=50),
    "winsorise": o.clip(lo, hi),
    "is_outlier": o > 50,
})

10% / 90% quantiles: 9.4 124.80000000000001


,o,clip(upper=50),winsorise,is_outlier
0,10,10,10.0,False
1,11,11,11.0,False
2,200,50,124.8,True
3,12,12,12.0,False
4,9,9,9.4,False


Clipping changes the data; a flag column keeps it and lets the model (or you) decide.
For prices, spikes are real events, so flagging is usually right.

In [40]:
df["price_spike"] = df["price_eur_mwh"] > df["price_eur_mwh"].quantile(0.99)
df["price_spike"].sum()

175

## 11. Final checks — make them `assert`s

An `assert` that passes prints nothing; one that fails stops the notebook. That is the point.

In [41]:
assert df.index.is_unique
assert df.index.is_monotonic_increasing
assert (df.index.to_series().diff().dropna() == pd.Timedelta("1h")).all()
assert df["price_eur_mwh"].dtype == "float64"
assert (df["temp_c"].dropna() > -100).all()
print("all checks passed;", len(df), "rows")

all checks passed; 17520 rows


## 12. Compare against the known-good file

Join on time and look at the largest absolute difference per column.

In [42]:
clean = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
both = df.join(clean, lsuffix="_mine", rsuffix="_clean")
both[["consumption_mwh_mine", "consumption_mwh_clean"]].head(3)

,consumption_mwh_mine,consumption_mwh_clean
time,,
2022-01-01 00:00:00+00:00,26858.4,26858.4
2022-01-01 01:00:00+00:00,26177.8,26177.8
2022-01-01 02:00:00+00:00,26229.4,26229.4


In [43]:
for col in ["consumption_mwh", "temp_c", "wind_ms", "price_eur_mwh"]:
    diff = (both[col + "_mine"] - both[col + "_clean"]).abs()
    print(f"{col:16s} max abs diff = {diff.max():.3f}  (rows compared: {diff.notna().sum()})")

consumption_mwh  max abs diff = 0.000  (rows compared: 17442)
temp_c           max abs diff = 3.890  (rows compared: 17520)
wind_ms          max abs diff = 0.000  (rows compared: 17442)
price_eur_mwh    max abs diff = 73.880  (rows compared: 17498)


Consumption and wind match exactly. Temperature and price differ only in the rows that
were NaN or a sentinel in the raw file and that we filled: interpolated temperatures are
close, but a forward-filled price copies a stale value and can be far off. That is the
cost of filling; a model would usually be better off with those rows dropped.

## 13. String clean-up on a tabular file

`strip` removes spaces, `lower` / `title` fix casing, `replace` maps spellings.

In [44]:
s = pd.Series(["London ", "london", "LONDON", "Wales"])
pd.DataFrame({"raw": s, "strip": s.str.strip(), "lower": s.str.strip().str.lower(), "title": s.str.strip().str.title()})

,raw,strip,lower,title
0,London,London,london,London
1,london,london,london,London
2,LONDON,LONDON,london,London
3,Wales,Wales,wales,Wales


In [45]:
meters = pd.read_csv("../data/meters.csv")
meters["region"].value_counts()

region
London      96
North       65
Scotland    57
Midlands    44
Wales       32
london       3
wales        1
north        1
midlands     1
Name: count, dtype: int64

The lowercase variants are the same regions. Normalise, then count again.

In [46]:
meters["region"] = meters["region"].str.strip().str.title()
meters["region"].value_counts()

region
London      99
North       66
Scotland    57
Midlands    45
Wales       33
Name: count, dtype: int64

In [47]:
meters["tariff"].replace({"TOU": "Time of use"}).value_counts(dropna=False)

tariff
Fixed          143
Variable        94
Time of use     50
NaN             13
Name: count, dtype: int64

## 14. Booleans read as text, dates as text, categoricals

A column of `"True"` / `"False"` strings is not boolean. Map it. Dates need `to_datetime`.
Low-cardinality strings can become `category` to save memory.

In [48]:
b = pd.Series(["True", "False", "True"])
print(b.dtype)
b.map({"True": True, "False": False})

object


0     True
1    False
2     True
dtype: bool

In [49]:
print(meters["has_solar"].dtype)          # pandas already parsed True/False here
meters["signup_date"] = pd.to_datetime(meters["signup_date"])
meters["signup_date"].dt.year.value_counts()

bool


signup_date
2021    158
2022    142
Name: count, dtype: int64

In [50]:
print("object  :", meters["region"].memory_usage(deep=True), "bytes")
print("category:", meters["region"].astype("category").memory_usage(deep=True), "bytes")

object  : 19133 bytes
category: 917 bytes


## The cleaning recipe (order matters)

1. parse timestamps (`utc=True`), sort
2. drop duplicates by key
3. text numbers → `to_numeric(errors="coerce")`, count what became NaN
4. sentinels → NaN
5. drop constant columns
6. reindex to the full grid, list the gaps
7. fill features carefully (limit!), never the target
8. clip / winsorise / flag outliers deliberately
9. `assert` the invariants
10. compare with a known-good source if there is one